# Train Qwen3 1.7B LoRA Adapters

Train local LoRA adapters for the current formula-direct and algorithmic-scaffold teaching variants.


In [13]:
!pip install -q -U "mlx-lm[train]" pandas tqdm

In [14]:
from pathlib import Path
import json
import shlex
import shutil
import subprocess
import sys

In [15]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune')

In [16]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
DATA_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"
RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"

In [17]:
ITERS = 1500
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
NUM_LAYERS = 12
LEARNING_RATE = "1e-5"
RESET_ADAPTERS_BEFORE_TRAINING = True
SKIP_IF_ADAPTER_EXISTS = False


In [18]:
EXPERIMENTS = {
    "formula_direct": {
        "data_dir": DATA_ROOT / "formula_direct",
        "adapter_path": RESULT_ROOT / "formula_direct" / "adapters",
    },
    "algorithmic_scaffold": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold" / "adapters",
    },
    "algorithmic_scaffold_v2": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v2",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v2" / "adapters",
    },
    "algorithmic_scaffold_v2_5": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v2_5",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v2_5" / "adapters",
    },
    "algorithmic_scaffold_v3": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v3",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v3" / "adapters",
    },
    "algorithmic_scaffold_v3_1": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v3_1",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v3_1" / "adapters",
    },
    "algorithmic_scaffold_v3_1_lora8": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v3_1",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v3_1_lora8" / "adapters",
    },
    "algorithmic_scaffold_v3_5": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v3_5",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v3_5" / "adapters",
    },
    "algorithmic_scaffold_v3_5_lora12": {
        "data_dir": DATA_ROOT / "algorithmic_scaffold_v3_5",
        "adapter_path": RESULT_ROOT / "algorithmic_scaffold_v3_5_lora12" / "adapters",
    },
}

# Keep this list small while iterating locally. Explicit/implicit theorem adapters
# are intentionally suppressed for now to save runtime.
EXPERIMENTS_TO_RUN = ["algorithmic_scaffold_v3_5_lora12"]
ACTIVE_EXPERIMENTS = {name: EXPERIMENTS[name] for name in EXPERIMENTS_TO_RUN}


In [19]:
def make_lora_command(data_dir, adapter_path):
    return [
        sys.executable,
        "-m",
        "mlx_lm.lora",
        "--model",
        MODEL_NAME,
        "--train",
        "--data",
        str(data_dir),
        "--adapter-path",
        str(adapter_path),
        "--iters",
        str(ITERS),
        "--batch-size",
        str(BATCH_SIZE),
        "--grad-accumulation-steps",
        str(GRAD_ACCUMULATION_STEPS),
        "--num-layers",
        str(NUM_LAYERS),
        "--learning-rate",
        LEARNING_RATE,
        "--mask-prompt",
        "--grad-checkpoint",
    ]

In [20]:
def adapter_exists(adapter_path):
    return (adapter_path / "adapters.safetensors").exists() or (adapter_path / "adapters.npz").exists()

In [21]:
def reset_adapter_dir(config):
    if config["adapter_path"].exists():
        shutil.rmtree(config["adapter_path"])



In [22]:
def run_lora_training(name, config):
    if SKIP_IF_ADAPTER_EXISTS and adapter_exists(config["adapter_path"]):
        print(f"Skipping {name}; adapter already exists at {config['adapter_path']}")
        return

    config["adapter_path"].mkdir(parents=True, exist_ok=True)
    command = make_lora_command(config["data_dir"], config["adapter_path"])
    print(shlex.join(command))
    subprocess.run(command, check=True)

## Smoke Train

Run this first if you want a quick command check for the first active adapter.


In [23]:
RUN_SMOKE = False

if RUN_SMOKE:
    smoke_name = EXPERIMENTS_TO_RUN[0]
    run_lora_training(smoke_name, EXPERIMENTS[smoke_name])


## Active Training Run

Runs only the adapters listed in `EXPERIMENTS_TO_RUN`.


In [24]:
RUN_TRAINING = True

if RUN_TRAINING:
    for name, config in ACTIVE_EXPERIMENTS.items():
        run_lora_training(name, config)


/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/.venv/bin/python -m mlx_lm.lora --model Qwen/Qwen3-1.7B-MLX-bf16 --train --data /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/algorithmic_scaffold_v3_5 --adapter-path /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters --iters 1500 --batch-size 1 --grad-accumulation-steps 8 --num-layers 12 --learning-rate 1e-5 --mask-prompt --grad-checkpoint
Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading pretrained model


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 90617.68it/s]


Loading datasets
Training
Trainable parameters: 0.217% (3.736M/1720.575M)
Starting training..., iters: 1500


Calculating loss...: 100%|██████████| 25/25 [00:13<00:00,  1.79it/s]


Iter 1: Val loss 3.723, Val took 13.950s
Iter 10: Train loss 3.234, Learning Rate 1.000e-05, It/sec 0.546, Tokens/sec 134.185, Trained Tokens 2459, Peak mem 4.594 GB
Iter 20: Train loss 2.514, Learning Rate 1.000e-05, It/sec 0.460, Tokens/sec 129.755, Trained Tokens 5277, Peak mem 4.607 GB
Iter 30: Train loss 2.431, Learning Rate 1.000e-05, It/sec 0.589, Tokens/sec 123.244, Trained Tokens 7370, Peak mem 4.609 GB
Iter 40: Train loss 1.917, Learning Rate 1.000e-05, It/sec 0.789, Tokens/sec 123.827, Trained Tokens 8940, Peak mem 4.609 GB
Iter 50: Train loss 2.093, Learning Rate 1.000e-05, It/sec 0.833, Tokens/sec 98.558, Trained Tokens 10123, Peak mem 4.609 GB
Iter 60: Train loss 1.549, Learning Rate 1.000e-05, It/sec 0.598, Tokens/sec 110.629, Trained Tokens 11973, Peak mem 4.609 GB
Iter 70: Train loss 1.303, Learning Rate 1.000e-05, It/sec 0.650, Tokens/sec 111.024, Trained Tokens 13682, Peak mem 4.609 GB
Iter 80: Train loss 1.236, Learning Rate 1.000e-05, It/sec 0.732, Tokens/sec 122.8

...

Iter 1200: Val loss 0.000, Val took 14.774s
Iter 1200: Train loss 0.003, Learning Rate 1.000e-05, It/sec 0.740, Tokens/sec 139.406, Trained Tokens 231664, Peak mem 4.624 GB
Iter 1200: Saved adapter weights to /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/adapters.safetensors and /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/0001200_adapters.safetensors.
Iter 1210: Train loss 0.000, Learning Rate 1.000e-05, It/sec 0.749, Tokens/sec 133.870, Trained Tokens 233452, Peak mem 4.624 GB
Iter 1220: Train loss 0.006, Learning Rate 1.000e-05, It/sec 0.639, Tokens/sec 141.632, Trained Tokens 235670, Peak mem 4.624 GB
Iter 1230: Train loss 0.003, Learning Rate 1.000e-05, It/sec 0.770, Tokens/sec 138.912, Trained Tokens 237475, Peak mem 4.624 GB
Iter 1240: Train loss 0.000, Learning Rate 1.000e-05, It/sec 0.652, Tokens/sec 12

Calculating loss...: 100%|██████████| 25/25 [00:15<00:00,  1.58it/s]


Iter 1400: Val loss 0.001, Val took 15.790s
Iter 1400: Train loss 0.002, Learning Rate 1.000e-05, It/sec 0.770, Tokens/sec 133.968, Trained Tokens 271087, Peak mem 4.624 GB
Iter 1400: Saved adapter weights to /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/adapters.safetensors and /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/0001400_adapters.safetensors.
Iter 1410: Train loss 0.000, Learning Rate 1.000e-05, It/sec 0.917, Tokens/sec 109.935, Trained Tokens 272286, Peak mem 4.624 GB
Iter 1420: Train loss 0.005, Learning Rate 1.000e-05, It/sec 0.670, Tokens/sec 139.516, Trained Tokens 274368, Peak mem 4.624 GB
Iter 1430: Train loss 0.000, Learning Rate 1.000e-05, It/sec 0.683, Tokens/sec 113.464, Trained Tokens 276029, Peak mem 4.624 GB
Iter 1440: Train loss 0.002, Learning Rate 1.000e-05, It/sec 0.641, Tokens/sec 14

Calculating loss...: 100%|██████████| 25/25 [00:18<00:00,  1.35it/s]


Iter 1500: Val loss 0.013, Val took 18.507s
Iter 1500: Train loss 0.000, Learning Rate 1.000e-05, It/sec 0.808, Tokens/sec 110.913, Trained Tokens 289473, Peak mem 4.624 GB
Iter 1500: Saved adapter weights to /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/adapters.safetensors and /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/0001500_adapters.safetensors.
Saved final weights to /Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/fine_tunes/qwen3_1_7b_lora/algorithmic_scaffold_v3_5_lora12/adapters/adapters.safetensors.


## Validation Accuracy

Use these cells after training to evaluate the adapters on validation data. This is the right split for choosing LoRA hyperparameters. The frozen `benchmark/data/test` split should be used only after the setup is chosen.

Validation is closed-book: the prompt contains only the problem, not the training reasoning. Metrics are reported separately for binary true/false-style tasks and non-binary short-answer tasks.

In [25]:
from mlx_lm import generate, load
from tqdm.auto import tqdm

/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
import importlib
import training_eval.eval_utils as eval_utils

importlib.reload(eval_utils)

GRADING_POLICY = eval_utils.GRADING_POLICY
balanced_eval_subset = eval_utils.balanced_eval_subset
extract_answer = eval_utils.extract_answer
extract_json_object = eval_utils.extract_json_object
is_correct = eval_utils.is_correct
load_jsonl_records = eval_utils.load_jsonl_records
rows_to_frame = eval_utils.rows_to_frame
save_results = eval_utils.save_results
summarize_accuracy = eval_utils.summarize_accuracy


In [27]:
VAL_RECORDS = {
    "formula_direct": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_formula_direct"),
    "algorithmic_scaffold": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold"),
    "algorithmic_scaffold_v2": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v2"),
    "algorithmic_scaffold_v2_5": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v2_5"),
    "algorithmic_scaffold_v3": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3"),
    "algorithmic_scaffold_v3_1": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_1"),
    "algorithmic_scaffold_v3_1_lora8": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_1"),
    "algorithmic_scaffold_v3_5": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_5"),
    "algorithmic_scaffold_v3_5_lora12": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_5"),
}

VALIDATION_RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_validation"
VALIDATION_LIMIT = None
RESET_VALIDATION_OUTPUTS = True
MAX_NEW_TOKENS = 512


In [28]:
FINE_TUNE_SYSTEM_MESSAGE = 'You solve discrete stochastic-process problems. Give concise reasoning, then end with exactly one final answer block: Final answer:\n<answer>\n{...}\n</answer>. Do not write anything after </answer>.'


In [29]:
def make_fine_tuned_chat_messages(problem):
    return [
        {"role": "system", "content": FINE_TUNE_SYSTEM_MESSAGE},
        {"role": "user", "content": problem},
    ]


In [30]:
def make_qwen_prompt(tokenizer, problem):
    messages = make_fine_tuned_chat_messages(problem)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [31]:
def load_adapter_model(adapter_path):
    return load(MODEL_NAME, adapter_path=str(adapter_path))

In [32]:
def generate_validation_answer(model, tokenizer, problem):
    prompt = make_qwen_prompt(tokenizer, problem)
    return generate(model, tokenizer, prompt=prompt, max_tokens=MAX_NEW_TOKENS, verbose=False)

In [33]:
def validation_result_dir(adapter_label):
    suffix = "full" if VALIDATION_LIMIT is None else f"subset_{VALIDATION_LIMIT}"
    return VALIDATION_RESULT_ROOT / f"{adapter_label}_{suffix}"


In [34]:
def validation_outputs_path(adapter_label):
    return validation_result_dir(adapter_label) / "outputs.jsonl"


In [35]:
def reset_validation_outputs(adapter_label):
    result_dir = validation_result_dir(adapter_label)
    for filename in ["outputs.jsonl", "outputs.csv", "metrics.json"]:
        path = result_dir / filename
        if path.exists():
            path.unlink()


In [36]:
def load_validation_cached_rows(adapter_label):
    path = validation_outputs_path(adapter_label)
    if not path.exists():
        return []
    rows_by_id = {}
    with path.open() as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                rows_by_id[row["id"]] = row
    return list(rows_by_id.values())


In [37]:
def append_validation_row(adapter_label, row):
    result_dir = validation_result_dir(adapter_label)
    result_dir.mkdir(parents=True, exist_ok=True)
    with validation_outputs_path(adapter_label).open("a") as f:
        f.write(json.dumps(row, sort_keys=True) + "\n")


In [38]:
def evaluate_adapter_on_validation(adapter_label, config):
    if RESET_VALIDATION_OUTPUTS:
        reset_validation_outputs(adapter_label)

    model, tokenizer = load_adapter_model(config["adapter_path"])
    records = balanced_eval_subset(VAL_RECORDS[adapter_label], VALIDATION_LIMIT)
    rows = load_validation_cached_rows(adapter_label)
    completed_ids = {row["id"] for row in rows}

    for record in tqdm(records, desc=f"val/{adapter_label}"):
        if record["id"] in completed_ids:
            continue

        raw_output = generate_validation_answer(model, tokenizer, record["problem"])
        predicted = extract_answer(raw_output, record["canonical_answer"])
        metadata = record.get("metadata", {})

        row = {
            "adapter": adapter_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "manual_variation": metadata.get("manual_variation", False),
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        }
        append_validation_row(adapter_label, row)
        rows.append(row)

    return rows


In [39]:
def save_validation_results(adapter_label, rows):
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "adapter": adapter_label,
        "dataset": "benchmark/data/val or benchmark/data/val_implicit_theorems",
        "validation_limit": VALIDATION_LIMIT,
        "split_role": "validation_for_hyperparameter_selection",
        "grading_policy": GRADING_POLICY,
    })
    return save_results(rows, validation_result_dir(adapter_label), metrics)

In [40]:
validation_rows = []

for adapter_label, config in ACTIVE_EXPERIMENTS.items():
    rows = evaluate_adapter_on_validation(adapter_label, config)
    save_validation_results(adapter_label, rows)
    validation_rows.extend(rows)

val_df = rows_to_frame(validation_rows)
val_df.head()

val/algorithmic_scaffold_v3_5_lora12: 100%|██████████| 240/240 [31:12<00:00,  7.80s/it]


,adapter,id,family,problem_type,difficulty,answer_type,manual_variation,problem,canonical_answer,raw_output,predicted_answer,correct
0,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_441000,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,True,Consider a simple symmetric random walk (X_n) ...,{'expected_time': '9'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '9'},True
1,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_441001,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,False,Let (S_n) be a simple symmetric random walk on...,{'expected_time': '3'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '3'},True
2,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_441002,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,False,Let (X_n) be a simple symmetric random walk on...,{'expected_time': '16'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '16'},True
3,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_441003,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,False,Let (S_n) be a simple symmetric random walk on...,{'expected_time': '12'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '12'},True
4,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_441004,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,False,Let (X_n) be a simple symmetric random walk on...,{'expected_time': '8'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '8'},True


In [41]:
display(val_df.groupby("adapter")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "answer_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "family"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "difficulty"])["correct"].agg(["mean", "sum", "count"]).sort_index())

,mean,sum,count
adapter,,,
algorithmic_scaffold_v3_5_lora12,0.920833,221,240


mean  sum  count
adapter                          answer_type                      
algorithmic_scaffold_v3_5_lora12 binary       0.900000  108    120
                                 non_binary   0.941667  113    120

mean  sum  \
adapter                          family                                       
algorithmic_scaffold_v3_5_lora12 hitting_time_expectation     0.883333   53   
                                 martingale_verification      0.950000   57   
                                 optional_stopping_validity   0.850000   51   
                                 stopped_process_expectation  1.000000   60   

                                                              count  
adapter                          family                              
algorithmic_scaffold_v3_5_lora12 hitting_time_expectation        60  
                                 martingale_verification         60  
                                 optional_stopping_validity      60  
                                 stopped_process_expectation     60

mean  sum  count
adapter                          difficulty                    
algorithmic_scaffold_v3_5_lora12 1           0.9375   75     80
                                 2           0.9375   75     80
                                 3           0.8875   71     80

In [42]:
val_df.loc[
    ~val_df["correct"],
    ["adapter", "id", "answer_type", "family", "problem_type", "difficulty", "canonical_answer", "predicted_answer", "raw_output"],
].head(30)

,adapter,id,answer_type,family,problem_type,difficulty,canonical_answer,predicted_answer,raw_output
25,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_442005,non_binary,hitting_time_expectation,symmetric_shifted_boundaries,2,{'expected_time': '15'},{'expected_time': '20'},Step 1: Identify the walk type.\nThe walk is s...
40,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443000,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '11/5'},{'expected_time': '67/5'},Step 1: Classify the walk and choose the biase...
41,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443001,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '43/7'},{'expected_time': '69/7'},Step 1: Classify the walk and choose the biase...
50,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443010,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '147/31'},{'expected_time': '1356'},Step 1: Classify the walk and choose the biase...
52,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443012,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '33/13'},{'expected_time': '-18985/1435'},Step 1: Classify the walk and choose the biase...
54,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443014,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '14990/2059'},{'expected_time': '15000/2059'},Step 1: Classify the walk and choose the biase...
57,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_val_443017,non_binary,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '1254/127'},{'expected_time': '636/127'},Step 1: Classify the walk and choose the biase...
65,algorithmic_scaffold_v3_5_lora12,martingale_verification_val_411005,binary,martingale_verification,centered_walk_basic,1,{'is_martingale': True},{'type': 'string'},Step 1: Identify the martingale test.\nCheck w...
75,algorithmic_scaffold_v3_5_lora12,martingale_verification_val_411015,binary,martingale_verification,centered_walk_basic,1,{'is_martingale': True},"{'tag': 'is_martingale', 'value': True}",Step 1: Identify the martingale test.\nCheck w...
85,algorithmic_scaffold_v3_5_lora12,martingale_verification_val_412005,binary,martingale_verification,quadratic_compensation,2,{'is_martingale': True},"{'prop_c': '25', 'req_c': '25'}",Step 1: Use only the compensation comparison.\...


## Train Generation Diagnostic

Use this to distinguish overfitting from underfitting. It evaluates the trained adapters by generation on a balanced sample from the training split, using the same prompt and grader as validation. If train generation is also weak, the adapter is not yet learning the task; if train generation is strong but validation is weak, it is overfitting.


In [43]:
TRAIN_RECORDS = {
    "formula_direct": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_formula_direct"),
    "algorithmic_scaffold": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold"),
    "algorithmic_scaffold_v2": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v2"),
    "algorithmic_scaffold_v2_5": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v2_5"),
    "algorithmic_scaffold_v3": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3"),
    "algorithmic_scaffold_v3_1": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_1"),
    "algorithmic_scaffold_v3_1_lora8": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_1"),
    "algorithmic_scaffold_v3_5": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_5"),
    "algorithmic_scaffold_v3_5_lora12": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_5"),
}

TRAIN_DIAGNOSTIC_RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_train_diagnostic"
TRAIN_DIAGNOSTIC_LIMIT = 120
RESET_TRAIN_DIAGNOSTIC_OUTPUTS = True


In [44]:
def train_diagnostic_result_dir(adapter_label):
    suffix = "full" if TRAIN_DIAGNOSTIC_LIMIT is None else f"subset_{TRAIN_DIAGNOSTIC_LIMIT}"
    return TRAIN_DIAGNOSTIC_RESULT_ROOT / f"{adapter_label}_{suffix}"


In [45]:
def train_diagnostic_outputs_path(adapter_label):
    return train_diagnostic_result_dir(adapter_label) / "outputs.jsonl"


In [46]:
def reset_train_diagnostic_outputs(adapter_label):
    result_dir = train_diagnostic_result_dir(adapter_label)
    for filename in ["outputs.jsonl", "outputs.csv", "metrics.json"]:
        path = result_dir / filename
        if path.exists():
            path.unlink()


In [47]:
def load_train_diagnostic_cached_rows(adapter_label):
    path = train_diagnostic_outputs_path(adapter_label)
    if not path.exists():
        return []
    rows_by_id = {}
    with path.open() as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                rows_by_id[row["id"]] = row
    return list(rows_by_id.values())


In [48]:
def append_train_diagnostic_row(adapter_label, row):
    result_dir = train_diagnostic_result_dir(adapter_label)
    result_dir.mkdir(parents=True, exist_ok=True)
    with train_diagnostic_outputs_path(adapter_label).open("a") as f:
        f.write(json.dumps(row, sort_keys=True) + "\n")


In [49]:
def evaluate_adapter_on_train_diagnostic(adapter_label, config):
    if RESET_TRAIN_DIAGNOSTIC_OUTPUTS:
        reset_train_diagnostic_outputs(adapter_label)

    model, tokenizer = load_adapter_model(config["adapter_path"])
    records = balanced_eval_subset(TRAIN_RECORDS[adapter_label], TRAIN_DIAGNOSTIC_LIMIT)
    rows = load_train_diagnostic_cached_rows(adapter_label)
    completed_ids = {row["id"] for row in rows}

    for record in tqdm(records, desc=f"train/{adapter_label}"):
        if record["id"] in completed_ids:
            continue

        raw_output = generate_validation_answer(model, tokenizer, record["problem"])
        predicted = extract_answer(raw_output, record["canonical_answer"])
        metadata = record.get("metadata", {})

        row = {
            "adapter": adapter_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "manual_variation": metadata.get("manual_variation", False),
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        }
        append_train_diagnostic_row(adapter_label, row)
        rows.append(row)

    return rows


In [50]:
def save_train_diagnostic_results(adapter_label, rows):
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "adapter": adapter_label,
        "dataset": "benchmark/data/train or benchmark/data/train_implicit_theorems",
        "train_diagnostic_limit": TRAIN_DIAGNOSTIC_LIMIT,
        "split_role": "train_generation_diagnostic",
        "grading_policy": GRADING_POLICY,
    })
    return save_results(rows, train_diagnostic_result_dir(adapter_label), metrics)


In [51]:
RUN_TRAIN_DIAGNOSTIC = True

train_diagnostic_rows = []

if RUN_TRAIN_DIAGNOSTIC:
    for adapter_label, config in ACTIVE_EXPERIMENTS.items():
        rows = evaluate_adapter_on_train_diagnostic(adapter_label, config)
        save_train_diagnostic_results(adapter_label, rows)
        train_diagnostic_rows.extend(rows)

train_df = rows_to_frame(train_diagnostic_rows)
train_df.head()


train/algorithmic_scaffold_v3_5_lora12: 100%|██████████| 120/120 [16:25<00:00,  8.21s/it]


,adapter,id,family,problem_type,difficulty,answer_type,manual_variation,problem,canonical_answer,raw_output,predicted_answer,correct
0,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_train_5220006,hitting_time_expectation,biased_boundaries_zero_a,3,non_binary,False,Let (S_n) be a nearest-neighbor random walk on...,{'expected_time': '1100/133'},Step 1: Classify the walk and choose the biase...,{'expected_time': '1100/133'},True
1,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_train_5200000,hitting_time_expectation,symmetric_boundaries_zero_a,1,non_binary,False,Let (S_n) be a simple symmetric random walk on...,{'expected_time': '32'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '32'},True
2,algorithmic_scaffold_v3_5_lora12,hitting_time_expectation_train_5210000,hitting_time_expectation,symmetric_shifted_boundaries,2,non_binary,False,Let (S_n) be a simple symmetric random walk wi...,{'expected_time': '4'},Step 1: Identify the walk type.\nThe walk is s...,{'expected_time': '4'},True
3,algorithmic_scaffold_v3_5_lora12,martingale_verification_train_5600000,martingale_verification,centered_walk_basic,1,binary,False,"Let S_0 = 0 and S_n = Y_1 + ... + Y_n, where t...",{'is_martingale': True},Step 1: Identify the martingale test.\nCheck w...,{'is_martingale': True},True
4,algorithmic_scaffold_v3_5_lora12,martingale_verification_train_5620003,martingale_verification,exponential_compensation,3,binary,False,"Let S_n = Y_1 + ... + Y_n, where the Y_k are i...",{'is_martingale': False},Step 1: Use only the denominator-factor compar...,{'is_martingale': False},True


In [52]:
display(train_df.groupby("adapter")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(train_df.groupby(["adapter", "answer_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(train_df.groupby(["adapter", "family"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(train_df.groupby(["adapter", "problem_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())


,mean,sum,count
adapter,,,
algorithmic_scaffold_v3_5_lora12,0.975,117,120


mean  sum  count
adapter                          answer_type                  
algorithmic_scaffold_v3_5_lora12 binary       1.00   60     60
                                 non_binary   0.95   57     60

mean  sum  count
adapter                          family                                       
algorithmic_scaffold_v3_5_lora12 hitting_time_expectation      0.9   27     30
                                 martingale_verification       1.0   30     30
                                 optional_stopping_validity    1.0   30     30
                                 stopped_process_expectation   1.0   30     30

mean  sum  \
adapter                          problem_type                              
algorithmic_scaffold_v3_5_lora12 biased_boundaries_zero_a       0.7    7   
                                 bounded_time_valid             1.0   10   
                                 bounded_walk_expectation       1.0   10   
                                 centered_walk_basic            1.0   10   
                                 exponential_compensation       1.0   10   
                                 finite_state_hitting_valid     1.0   10   
                                 quadratic_compensation         1.0   10   
                                 quadratic_fixed_horizon        1.0   10   
                                 stopped_martingale_value       1.0   10   
                                 symmetric_boundaries_zero_a    1.0   10   
                                 symmetric_shifted_boundaries   1.0   10   
                                 unbounded_first_hit_invalid    1.0   10   

                                                               count  
adapter                          problem_type                         
algorithmic_scaffold_v3_5_lora12 biased_boundaries_zero_a         10  
                                 bounded_time_valid               10  
                                 bounded_walk_expectation         10  
                                 centered_walk_basic              10  
                                 exponential_compensation         10  
                                 finite_state_hitting_valid       10  
                                 quadratic_compensation           10  
                                 quadratic_fixed_horizon          10  
                                 stopped_martingale_value         10  
                                 symmetric_boundaries_zero_a      10  
                                 symmetric_shifted_boundaries     10  
                                 unbounded_first_hit_invalid      10